In [1]:
import numpy as np
import sys
import matplotlib.pyplot as plt
from IPython.display import clear_output
from stable_baselines3 import SAC
import requests
import csv
import torch
import datetime
import os
%load_ext tensorboard
import tensorflow as tf

In [2]:
clear_output(wait=True)

In [3]:
url = "http://localhost:80"
log_path = os.path.join("Logs")

In [4]:
sys.path.insert(0,'../boptestGym')
from boptestGymEnv import BoptestGymEnv
from boptestGymEnv import NormalizedObservationWrapper

In [5]:
kelvin = lambda c: c + 273.15

bounds = {
    "T_in": (273,324),
    "T_out": (257,305),
    "Psol": (0, 862),
}

step_period_h = 0.5
floor_area = 16 * 12
low_setpoint_k  = kelvin(21.0)
high_setpoint_k = kelvin(24.0)

In [6]:
class BoptestGymEnvCustomReward(BoptestGymEnv):
    def reset(self, *args, **kwargs):
        obs, info = super().reset(**kwargs)
        self.last_obs = obs
        self.last_action=0
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        self.last_obs = obs
        self.last_action = action
        return obs, reward, terminated, truncated, info
    
    def get_reward(self):
        obs = self.last_obs
        action = self.last_action
        T_in_K_val = obs[0]
        discomfort = (max(0.0, low_setpoint_k  - T_in_K_val) +
                  max(0.0, T_in_K_val - high_setpoint_k)) * step_period_h
        energy = obs[3] * step_period_h / (3600. * floor_area)
        reward = -(discomfort + energy)

        self.objective_integrand = reward
        return reward

In [7]:
feb15 = 47 * 24*3600                    # Jan 1 → Feb 15 low data regime
june30 = 181 * 24 *3600                 # high data regime
episode_length = 7 * 24*3600            # 1 week episodes

In [8]:
def create_env(max_start):
    env = BoptestGymEnvCustomReward(url  = url,
                    testcase              = 'bestest_hydronic_heat_pump',
                    actions               = ['oveTSet_u'],
                    observations          = {'reaTZon_y':(273.,324.), #0 / 50
                                             'weaSta_reaWeaTDryBul_y':(257.,305.), # -15 / 28.8
                                             'weaSta_reaWeaHDirNor_y':(0.,862.),
                                             'reaQFloHea_y':(0.,  15000.),
                                            },
                    random_start_time     = True,
                    max_episode_length    = 7 * 24*3600,
                    excluding_periods     = [(max_start, 365*24*3600)],
                    warmup_period         = 24*3600,
                    predictive_period     = 0,
                    regressive_period     = 4*1800,
                    step_period           = 1800
                               )
    env = NormalizedObservationWrapper(env)
    print("env created")
    return env

In [9]:
for data_regime in [6, 27]:
    for training_episodes in [25, 50]:
        print(f"training episode: {training_episodes}")
        if data_regime == 6:
            max_start = feb15 - episode_length      # last valid start time
            print("6 weeks")
        elif data_regime == 27:
            max_start = june30 - episode_length      # last valid start time
            print("27 weeks")
        env = create_env(max_start)    

        model = SAC(
            policy='MlpPolicy',
            env=env,
            verbose=1,
            learning_rate=1e-4,
            batch_size=1024,
            tau=0.005,
            gamma=0.99,
            tensorboard_log=log_path,
            seed=42,
        )
        
        total_timesteps = 336 * training_episodes 
        model.learn(total_timesteps= total_timesteps, tb_log_name = f"SAC_temp_1_{data_regime}_{training_episodes}") 
        model.save(f"Saved Models/SAC_temp_1_{data_regime}_{training_episodes}")
        print(f"model saved at: Saved Models/SAC_temp_1_{data_regime}_{training_episodes}")
        buffer_path = os.path.join( "Memory Buffers", f"SAC_temp_1_{data_regime}_{training_episodes}")
        model.save_replay_buffer(buffer_path)
        print("memory buffer saved")
        env.stop()
        del model

training episode: 25
6 weeks


C:\Users\irmak\anaconda3\Lib\site-packages\gymnasium\spaces\box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


env created
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to Logs\SAC_temp_1_6_25_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 336      |
|    ep_rew_mean     | -18.7    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 13       |
|    time_elapsed    | 97       |
|    total_timesteps | 1344     |
| train/             |          |
|    actor_loss      | -4.27    |
|    critic_loss     | 0.0233   |
|    ent_coef        | 0.883    |
|    ent_coef_loss   | -0.209   |
|    learning_rate   | 0.0001   |
|    n_updates       | 1243     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 336      |
|    ep_rew_mean     | -53.3    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 13       |
|    time_elapsed    | 194      |
|    tota

C:\Users\irmak\anaconda3\Lib\site-packages\gymnasium\spaces\box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


env created
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to Logs\SAC_temp_1_6_50_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 336      |
|    ep_rew_mean     | -18.7    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 14       |
|    time_elapsed    | 95       |
|    total_timesteps | 1344     |
| train/             |          |
|    actor_loss      | -4.27    |
|    critic_loss     | 0.0233   |
|    ent_coef        | 0.883    |
|    ent_coef_loss   | -0.209   |
|    learning_rate   | 0.0001   |
|    n_updates       | 1243     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 336      |
|    ep_rew_mean     | -53.3    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 14       |
|    time_elapsed    | 191      |
|    tota

In [10]:
env.stop()